# DenseNet121 baseline vs DenseNet121 + Coordinate Attention

Thí nghiệm đối chứng công bằng trên bộ dữ liệu X-quang ngực 4 lớp. Hai mô hình dùng cùng dữ liệu, split, augmentation, seed, batch size, optimizer, scheduler, freeze/fine-tune và early stopping. Khác biệt kiến trúc duy nhất là **Coordinate Attention (CA)** đặt sau `DenseNet121.features` + ReLU và trước global average pooling.

Notebook này được tạo từ việc đọc `DenseNet_train.ipynb`. Các thay đổi có chủ đích so với notebook gốc: thêm baseline/CA riêng biệt, reproducibility cho sampler/workers, AMP, macro-F1, checkpoint đầy đủ, sửa lỗi tái tạo scheduler sau khi tái tạo optimizer lúc unfreeze, đánh giá chung, reload verification và xuất báo cáo. Không dùng TTA, ensemble, MixUp, CutMix, label smoothing hoặc class-weighted loss.


## 1. Imports và cấu hình Kaggle


In [ ]:
import gc
import json
import os
import random
import shutil
import time
import warnings
from collections import Counter
from contextlib import nullcontext
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image, UnidentifiedImageError
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms
from tqdm.auto import tqdm

SEED = 42
DATASET_ROOT = "/kaggle/input/datasets/jtiptj/chest-xray-pneumoniacovid19tuberculosis"
IMAGE_SIZE = 300
BATCH_SIZE = 16
NUM_CLASSES = 4
NUM_EPOCHS = 50
FREEZE_EPOCHS = 5
LEARNING_RATE = 1e-4
FINE_TUNE_LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-4
PATIENCE = 8
CA_REDUCTION = 32
NUM_WORKERS = 2
USE_AMP = True

OUTPUT_DIR = Path("/kaggle/working/densenet_ca_comparison")
FIGURES_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = bool(USE_AMP and DEVICE.type == "cuda")

print("Device:", DEVICE)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    warnings.warn("Không có CUDA; notebook sẽ chạy FP32 trên CPU và có thể rất chậm.")
print("Output directory:", OUTPUT_DIR)


## 2. Tái lập kết quả


In [ ]:
def set_seed(seed: int = 42) -> None:
    """Thiết lập seed cho Python, NumPy, PyTorch CPU/CUDA và cuDNN."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(worker_id: int) -> None:
    """Seed NumPy và Python random trong từng DataLoader worker."""
    del worker_id
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_grad_scaler() -> torch.cuda.amp.GradScaler:
    try:
        return torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)


def autocast_context():
    if not AMP_ENABLED:
        return nullcontext()
    try:
        return torch.amp.autocast(device_type="cuda", dtype=torch.float16)
    except (AttributeError, TypeError):
        return torch.cuda.amp.autocast(dtype=torch.float16)


set_seed(SEED)


## 3. Kiểm tra split, xây records và kiểm tra file ảnh


In [ ]:
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
root = Path(DATASET_ROOT)
if not root.exists():
    raise FileNotFoundError(f"Không tìm thấy DATASET_ROOT: {root}")

split_paths = {split: root / split for split in ("train", "val", "test")}
missing_splits = [str(path) for path in split_paths.values() if not path.is_dir()]
if missing_splits:
    raise FileNotFoundError(f"Thiếu split bắt buộc: {missing_splits}")

class_names = sorted(path.name for path in split_paths["train"].iterdir() if path.is_dir())
if len(class_names) != NUM_CLASSES:
    raise ValueError(f"Cần {NUM_CLASSES} lớp trong train, tìm thấy {len(class_names)}: {class_names}")
class_to_idx = {name: idx for idx, name in enumerate(class_names)}

for split, split_path in split_paths.items():
    split_classes = sorted(path.name for path in split_path.iterdir() if path.is_dir())
    if split_classes != class_names:
        raise ValueError(
            f"Thứ tự/tập class của {split} không khớp train. "
            f"train={class_names}, {split}={split_classes}"
        )


def is_image_file(path: Path) -> bool:
    return path.is_file() and path.suffix.lower() in VALID_EXTENSIONS


def verify_image(path: Path) -> None:
    """Phát hiện sớm file ảnh lỗi thay vì lỗi giữa quá trình train."""
    try:
        with Image.open(path) as image:
            image.verify()
    except (UnidentifiedImageError, OSError, ValueError) as exc:
        raise RuntimeError(f"Ảnh lỗi hoặc không đọc được: {path}") from exc


records: Dict[str, List[Tuple[str, int]]] = {"train": [], "val": [], "test": []}
count_rows: List[Dict[str, Any]] = []
for split, split_path in split_paths.items():
    for class_name in class_names:
        class_path = split_path / class_name
        image_paths = sorted(path for path in class_path.rglob("*") if is_image_file(path))
        if not image_paths:
            raise ValueError(f"Không có ảnh hợp lệ trong: {class_path}")
        for image_path in tqdm(image_paths, desc=f"Verify {split}/{class_name}", leave=False):
            verify_image(image_path)
            records[split].append((str(image_path), class_to_idx[class_name]))
        count_rows.append({"split": split, "class": class_name, "count": len(image_paths)})

df_counts = pd.DataFrame(count_rows)
print("Class names:", class_names)
print("class_to_idx:", class_to_idx)
display(df_counts.pivot(index="class", columns="split", values="count"))
print("Total images:", {split: len(items) for split, items in records.items()})


## 4. Dataset, augmentation và DataLoader tái lập


In [ ]:
class CXRDataset(Dataset):
    """Dataset X-quang kế thừa cách đọc RGB từ notebook gốc."""

    def __init__(self, records: Sequence[Tuple[str, int]], transform=None):
        self.records = list(records)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        img_path, label = self.records[idx]
        try:
            with Image.open(img_path) as source:
                image = source.convert("RGB")
        except (UnidentifiedImageError, OSError, ValueError) as exc:
            raise RuntimeError(f"Không thể đọc ảnh tại index {idx}: {img_path}") from exc
        if self.transform is not None:
            image = self.transform(image)
        return image, int(label)


# Giữ nguyên preprocessing/augmentation của DenseNet_train.ipynb.
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = CXRDataset(records["train"], transform=train_transform)
val_dataset = CXRDataset(records["val"], transform=val_test_transform)
test_dataset = CXRDataset(records["test"], transform=val_test_transform)


def create_dataloaders(batch_size: int, seed: int = SEED) -> Tuple[DataLoader, DataLoader, DataLoader]:
    """Tạo sampler và loaders mới cho mỗi model với cùng seed."""
    train_labels = [label for _, label in records["train"]]
    class_counts = Counter(train_labels)
    class_weights = {label: 1.0 / count for label, count in class_counts.items()}
    sample_weights = torch.as_tensor(
        [class_weights[label] for label in train_labels], dtype=torch.double
    )

    sampler_generator = torch.Generator().manual_seed(seed)
    worker_generator = torch.Generator().manual_seed(seed)
    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True,
        generator=sampler_generator,
    )
    common = dict(
        batch_size=batch_size,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        worker_init_fn=seed_worker,
        persistent_workers=False,
    )
    train_loader = DataLoader(
        train_dataset, sampler=sampler, generator=worker_generator, **common
    )
    val_loader = DataLoader(
        val_dataset, shuffle=False, generator=torch.Generator().manual_seed(seed), **common
    )
    test_loader = DataLoader(
        test_dataset, shuffle=False, generator=torch.Generator().manual_seed(seed), **common
    )
    return train_loader, val_loader, test_loader


inspection_train_loader, inspection_val_loader, inspection_test_loader = create_dataloaders(BATCH_SIZE)
batch_images, batch_labels = next(iter(inspection_train_loader))
print("Image batch shape:", tuple(batch_images.shape))
print("Label shape:", tuple(batch_labels.shape))
print("Train class counts:", Counter(label for _, label in records["train"]))
del inspection_train_loader, inspection_val_loader, inspection_test_loader, batch_images, batch_labels


## 5. DenseNet121 baseline và Coordinate Attention


In [ ]:
def build_classifier(in_features: int, num_classes: int) -> nn.Sequential:
    """Classifier giống hệt notebook gốc cho cả hai mô hình."""
    return nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(in_features, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(0.3),
        nn.Linear(512, num_classes),
    )


class HSigmoid(nn.Module):
    def __init__(self, inplace: bool = True):
        super().__init__()
        self.relu = nn.ReLU6(inplace=inplace)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.relu(x + 3.0) / 6.0


class HSwish(nn.Module):
    def __init__(self, inplace: bool = True):
        super().__init__()
        self.h_sigmoid = HSigmoid(inplace=inplace)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * self.h_sigmoid(x)


class CoordinateAttention(nn.Module):
    """Coordinate Attention không làm thay đổi shape của feature map."""

    def __init__(self, in_channels: int, reduction: int = 32):
        super().__init__()
        mip = max(8, in_channels // reduction)
        self.conv1 = nn.Conv2d(in_channels, mip, kernel_size=1, bias=False)
        self.batch_norm = nn.BatchNorm2d(mip)
        self.h_swish = HSwish()
        self.conv_h = nn.Conv2d(mip, in_channels, kernel_size=1)
        self.conv_w = nn.Conv2d(mip, in_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x
        _, _, height, width = x.shape
        x_h = x.mean(dim=3, keepdim=True)                       # B,C,H,1
        x_w = x.mean(dim=2, keepdim=True).permute(0, 1, 3, 2) # B,C,W,1
        y = torch.cat([x_h, x_w], dim=2)
        y = self.h_swish(self.batch_norm(self.conv1(y)))
        y_h, y_w = torch.split(y, [height, width], dim=2)
        y_w = y_w.permute(0, 1, 3, 2)
        a_h = torch.sigmoid(self.conv_h(y_h))
        a_w = torch.sigmoid(self.conv_w(y_w))
        output = identity * a_h * a_w
        if output.shape != identity.shape:
            raise RuntimeError(f"CA shape mismatch: {identity.shape} -> {output.shape}")
        return output


class DenseNet121Baseline(nn.Module):
    """DenseNet121 baseline: features -> ReLU -> GAP -> classifier."""

    def __init__(self, num_classes: int = NUM_CLASSES, pretrained: bool = True):
        super().__init__()
        weights = models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = models.densenet121(weights=weights)
        self.features = backbone.features
        self.feature_channels = int(backbone.classifier.in_features)
        if self.feature_channels != 1024:
            raise ValueError(f"DenseNet121 feature channels không phải 1024: {self.feature_channels}")
        self.classifier = build_classifier(self.feature_channels, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = F.relu(self.features(x), inplace=True)
        pooled = F.adaptive_avg_pool2d(features, (1, 1))
        return self.classifier(torch.flatten(pooled, 1))


class DenseNet121WithCA(nn.Module):
    """DenseNet121 với CA sau ReLU cuối và ngay trước global average pooling."""

    def __init__(self, num_classes: int = NUM_CLASSES, pretrained: bool = True, reduction: int = 32):
        super().__init__()
        weights = models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = models.densenet121(weights=weights)
        self.features = backbone.features
        self.feature_channels = int(backbone.classifier.in_features)
        if self.feature_channels != 1024:
            raise ValueError(f"DenseNet121 feature channels không phải 1024: {self.feature_channels}")
        # Tạo classifier trước CA để classifier có cùng khởi tạo RNG như baseline.
        self.classifier = build_classifier(self.feature_channels, num_classes)
        self.coordinate_attention = CoordinateAttention(self.feature_channels, reduction=reduction)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = F.relu(self.features(x), inplace=True)
        features = self.coordinate_attention(features)  # CA nằm trước GAP.
        pooled = F.adaptive_avg_pool2d(features, (1, 1))
        return self.classifier(torch.flatten(pooled, 1))


def count_parameters(model: nn.Module) -> Tuple[int, int]:
    total = sum(parameter.numel() for parameter in model.parameters())
    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    return total, trainable


def has_coordinate_attention(model: nn.Module) -> bool:
    return any(isinstance(module, CoordinateAttention) for module in model.modules())


## 6. Smoke test kiến trúc và chọn batch size chung trước khi train


In [ ]:
set_seed(SEED)
baseline_smoke = DenseNet121Baseline(pretrained=True).to(DEVICE)
dummy = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
with torch.no_grad():
    baseline_output = baseline_smoke(dummy)
assert baseline_output.shape == (2, NUM_CLASSES)
assert not has_coordinate_attention(baseline_smoke)
baseline_total, baseline_trainable = count_parameters(baseline_smoke)
print("Baseline output shape:", tuple(baseline_output.shape))
print(f"Baseline parameters: total={baseline_total:,}, trainable={baseline_trainable:,}")
print("Baseline contains CA:", has_coordinate_attention(baseline_smoke))

# Giải phóng baseline khỏi GPU trước khi tạo CA, kể cả trong smoke test.
baseline_smoke = baseline_smoke.cpu()
del baseline_smoke, baseline_output, dummy
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

set_seed(SEED)
ca_smoke = DenseNet121WithCA(pretrained=True, reduction=CA_REDUCTION).to(DEVICE)
dummy = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
with torch.no_grad():
    ca_output = ca_smoke(dummy)
assert ca_output.shape == (2, NUM_CLASSES)
assert has_coordinate_attention(ca_smoke)
ca_total, ca_trainable = count_parameters(ca_smoke)
print("CA output shape:", tuple(ca_output.shape))
print(f"CA parameters: total={ca_total:,}, trainable={ca_trainable:,}")
print(f"Additional CA parameters: {ca_total - baseline_total:,}")
print("DenseNet121-CA contains CA:", has_coordinate_attention(ca_smoke))
for name, module in ca_smoke.named_modules():
    if isinstance(module, CoordinateAttention):
        print(f"CA module: {name} ({module.__class__.__name__}); vị trí: sau features+ReLU, trước GAP")


def resolve_common_batch_size(start_batch_size: int = 16) -> int:
    """Probe mô hình CA (lớn hơn) để chọn 16 hoặc 8 trước khi train cả hai model."""
    if DEVICE.type != "cuda":
        return start_batch_size
    probe_model = ca_smoke
    probe_model.train()
    for candidate in (start_batch_size, 8):
        try:
            probe_model.zero_grad(set_to_none=True)
            probe = torch.randn(candidate, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
            with autocast_context():
                loss = probe_model(probe).sum()
            loss.backward()
            del probe, loss
            probe_model.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()
            if candidate != start_batch_size:
                warnings.warn(
                    f"CUDA OOM với batch {start_batch_size}; cả hai model sẽ dùng batch {candidate}."
                )
            return candidate
        except RuntimeError as error:
            if "out of memory" not in str(error).lower():
                raise
            probe_model.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()
    raise RuntimeError("CUDA OOM cả với batch size 8; không thay đổi kiến trúc tự động.")


EFFECTIVE_BATCH_SIZE = resolve_common_batch_size(BATCH_SIZE)
print("Common effective batch size:", EFFECTIVE_BATCH_SIZE)

ca_smoke = ca_smoke.cpu()
del ca_smoke, dummy, ca_output
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 7. Hàm train/validation dùng chung


In [ ]:
def compute_epoch_metrics(labels: Sequence[int], predictions: Sequence[int]) -> Tuple[float, float]:
    accuracy = accuracy_score(labels, predictions)
    macro_f1 = f1_score(labels, predictions, average="macro", zero_division=0)
    return float(accuracy), float(macro_f1)


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: optim.Optimizer,
    criterion: nn.Module,
    scaler,
    backbone_frozen: bool,
) -> Dict[str, float]:
    """Train một epoch và trả loss, accuracy, macro-F1."""
    model.train()
    if backbone_frozen:
        model.features.eval()  # Không cập nhật running statistics của backbone đã freeze.
    running_loss = 0.0
    all_labels: List[int] = []
    all_predictions: List[int] = []

    for images, labels in tqdm(loader, desc="Train", leave=False):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast_context():
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        all_labels.extend(labels.detach().cpu().tolist())
        all_predictions.extend(outputs.detach().argmax(dim=1).cpu().tolist())

    accuracy, macro_f1 = compute_epoch_metrics(all_labels, all_predictions)
    return {"loss": running_loss / len(loader.dataset), "accuracy": accuracy, "macro_f1": macro_f1}


@torch.no_grad()
def validate_one_epoch(model: nn.Module, loader: DataLoader, criterion: nn.Module) -> Dict[str, float]:
    """Validate một epoch và trả loss, accuracy, macro-F1."""
    model.eval()
    running_loss = 0.0
    all_labels: List[int] = []
    all_predictions: List[int] = []
    for images, labels in tqdm(loader, desc="Validation", leave=False):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        with autocast_context():
            outputs = model(images)
            loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        all_labels.extend(labels.cpu().tolist())
        all_predictions.extend(outputs.argmax(dim=1).cpu().tolist())
    accuracy, macro_f1 = compute_epoch_metrics(all_labels, all_predictions)
    return {"loss": running_loss / len(loader.dataset), "accuracy": accuracy, "macro_f1": macro_f1}


def make_optimizer_scheduler(
    model: nn.Module, learning_rate: float
) -> Tuple[optim.Optimizer, optim.lr_scheduler.ReduceLROnPlateau]:
    optimizer = optim.AdamW(
        [parameter for parameter in model.parameters() if parameter.requires_grad],
        lr=learning_rate,
        weight_decay=WEIGHT_DECAY,
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.3, patience=3
    )
    return optimizer, scheduler


def make_config(coordinate_attention: bool) -> Dict[str, Any]:
    return {
        "batch_size": EFFECTIVE_BATCH_SIZE,
        "num_epochs": NUM_EPOCHS,
        "freeze_epochs": FREEZE_EPOCHS,
        "initial_learning_rate": LEARNING_RATE,
        "fine_tune_learning_rate": FINE_TUNE_LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "patience": PATIENCE,
        "coordinate_attention": coordinate_attention,
        "ca_reduction": CA_REDUCTION if coordinate_attention else None,
    }


## 8. Training loop, best checkpoint theo validation macro-F1


In [ ]:
def train_model(
    model: nn.Module,
    model_name: str,
    architecture: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    best_checkpoint_path: Path,
    history_path: Path,
    coordinate_attention: bool,
) -> Dict[str, Any]:
    """Train chung cho baseline/CA, freeze 5 epoch rồi fine-tune toàn bộ model."""
    criterion = nn.CrossEntropyLoss()
    for parameter in model.features.parameters():
        parameter.requires_grad = False
    optimizer, scheduler = make_optimizer_scheduler(model, LEARNING_RATE)
    scaler = make_grad_scaler()
    history: List[Dict[str, float]] = []
    best_val_macro_f1 = -float("inf")
    best_val_loss = float("inf")
    best_val_accuracy = 0.0
    best_epoch = 0
    early_stop_counter = 0
    start_time = time.time()
    config = make_config(coordinate_attention)

    for epoch_index in range(NUM_EPOCHS):
        epoch = epoch_index + 1
        if epoch_index == FREEZE_EPOCHS:
            print(f"\n[{model_name}] Unfreeze backbone và chuyển LR sang {FINE_TUNE_LEARNING_RATE:.1e}")
            for parameter in model.features.parameters():
                parameter.requires_grad = True
            # Bắt buộc tạo lại cả optimizer lẫn scheduler; scheduler không giữ optimizer cũ.
            optimizer, scheduler = make_optimizer_scheduler(model, FINE_TUNE_LEARNING_RATE)
            scaler = make_grad_scaler()
            early_stop_counter = 0

        backbone_frozen = epoch_index < FREEZE_EPOCHS
        learning_rate = float(optimizer.param_groups[0]["lr"])
        train_metrics = train_one_epoch(
            model, train_loader, optimizer, criterion, scaler, backbone_frozen
        )
        val_metrics = validate_one_epoch(model, val_loader, criterion)
        scheduler.step(val_metrics["loss"])

        row = {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_macro_f1": train_metrics["macro_f1"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "learning_rate": learning_rate,
        }
        history.append(row)
        pd.DataFrame(history).to_csv(history_path, index=False)

        f1_better = val_metrics["macro_f1"] > best_val_macro_f1 + 1e-12
        f1_tied_loss_better = (
            abs(val_metrics["macro_f1"] - best_val_macro_f1) <= 1e-12
            and val_metrics["loss"] < best_val_loss
        )
        improved = f1_better or f1_tied_loss_better
        if improved:
            best_val_macro_f1 = val_metrics["macro_f1"]
            best_val_loss = val_metrics["loss"]
            best_val_accuracy = val_metrics["accuracy"]
            best_epoch = epoch
            if not backbone_frozen:
                early_stop_counter = 0
            checkpoint = {
                "model_name": model_name,
                "architecture": architecture,
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_val_loss": best_val_loss,
                "best_val_accuracy": best_val_accuracy,
                "best_val_macro_f1": best_val_macro_f1,
                "class_names": class_names,
                "class_to_idx": class_to_idx,
                "image_size": IMAGE_SIZE,
                "num_classes": NUM_CLASSES,
                "seed": SEED,
                "history": history.copy(),
                "config": config,
            }
            torch.save(checkpoint, best_checkpoint_path)
            save_status = "saved best"
        else:
            if not backbone_frozen:
                early_stop_counter += 1
            save_status = f"no improve ({early_stop_counter}/{PATIENCE}, fine-tune only)"

        print(
            f"[{model_name}] Epoch {epoch:02d}/{NUM_EPOCHS} | "
            f"Train loss {row['train_loss']:.5f}, acc {row['train_accuracy']:.5f}, F1 {row['train_macro_f1']:.5f} | "
            f"Val loss {row['val_loss']:.5f}, acc {row['val_accuracy']:.5f}, F1 {row['val_macro_f1']:.5f} | "
            f"LR {learning_rate:.2e} | {save_status}"
        )
        if not backbone_frozen and early_stop_counter >= PATIENCE:
            print(f"[{model_name}] Early stopping; best epoch = {best_epoch}")
            break

    if not best_checkpoint_path.exists():
        raise RuntimeError(f"Không tạo được best checkpoint: {best_checkpoint_path}")
    training_minutes = (time.time() - start_time) / 60.0
    return {
        "history": history,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "best_val_accuracy": best_val_accuracy,
        "best_val_macro_f1": best_val_macro_f1,
        "training_time_minutes": training_minutes,
        "config": config,
    }


## 9. Đánh giá test, lưu final và reload model


In [ ]:
@torch.no_grad()
def evaluate_model(
    model: nn.Module, loader: DataLoader, criterion: nn.Module, class_names: Sequence[str]
) -> Dict[str, Any]:
    """Đánh giá chung: loss, metrics, report, confusion matrix, labels và probabilities."""
    model.eval()
    running_loss = 0.0
    true_labels: List[int] = []
    predicted_labels: List[int] = []
    predicted_probabilities: List[List[float]] = []
    for images, labels in tqdm(loader, desc="Test", leave=False):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        with autocast_context():
            outputs = model(images)
            loss = criterion(outputs, labels)
        probabilities = torch.softmax(outputs, dim=1)
        predictions = probabilities.argmax(dim=1)
        running_loss += loss.item() * images.size(0)
        true_labels.extend(labels.cpu().tolist())
        predicted_labels.extend(predictions.cpu().tolist())
        predicted_probabilities.extend(probabilities.cpu().tolist())

    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        true_labels, predicted_labels, average="macro", zero_division=0
    )
    weighted_f1 = f1_score(true_labels, predicted_labels, average="weighted", zero_division=0)
    report = classification_report(
        true_labels,
        predicted_labels,
        labels=list(range(len(class_names))),
        target_names=list(class_names),
        zero_division=0,
        output_dict=True,
    )
    matrix = confusion_matrix(
        true_labels, predicted_labels, labels=list(range(len(class_names)))
    )
    return {
        "loss": running_loss / len(loader.dataset),
        "accuracy": float(accuracy_score(true_labels, predicted_labels)),
        "macro_precision": float(macro_precision),
        "macro_recall": float(macro_recall),
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1),
        "classification_report": report,
        "confusion_matrix": matrix,
        "true_labels": np.asarray(true_labels),
        "predicted_labels": np.asarray(predicted_labels),
        "predicted_probabilities": np.asarray(predicted_probabilities),
    }


def save_and_evaluate_best(
    model: nn.Module,
    best_checkpoint_path: Path,
    final_checkpoint_path: Path,
    test_loader: DataLoader,
    training_result: Dict[str, Any],
) -> Dict[str, Any]:
    best_checkpoint = torch.load(best_checkpoint_path, map_location=DEVICE)
    model.load_state_dict(best_checkpoint["model_state_dict"])
    model.to(DEVICE).eval()
    evaluation = evaluate_model(model, test_loader, nn.CrossEntropyLoss(), class_names)
    test_metrics = {
        key: evaluation[key]
        for key in ("loss", "accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_f1")
    }
    final_checkpoint = {
        "model_name": best_checkpoint["model_name"],
        "architecture": best_checkpoint["architecture"],
        "model_state_dict": model.state_dict(),
        "best_epoch": best_checkpoint["epoch"],
        "class_names": class_names,
        "class_to_idx": class_to_idx,
        "image_size": IMAGE_SIZE,
        "num_classes": NUM_CLASSES,
        "test_metrics": test_metrics,
        "classification_report": evaluation["classification_report"],
        "confusion_matrix": evaluation["confusion_matrix"].tolist(),
        "history": training_result["history"],
        "config": training_result["config"],
    }
    torch.save(final_checkpoint, final_checkpoint_path)
    return evaluation


def load_trained_model(checkpoint_path: Path, device: torch.device):
    """Tự khởi tạo đúng architecture, load state_dict, chuyển device và eval."""
    checkpoint = torch.load(checkpoint_path, map_location=device)
    architecture = checkpoint.get("architecture")
    num_classes = int(checkpoint.get("num_classes", NUM_CLASSES))
    if architecture == "DenseNet121Baseline":
        model = DenseNet121Baseline(num_classes=num_classes, pretrained=False)
    elif architecture == "DenseNet121WithCA":
        reduction = int(checkpoint.get("config", {}).get("ca_reduction", CA_REDUCTION))
        model = DenseNet121WithCA(
            num_classes=num_classes, pretrained=False, reduction=reduction
        )
    else:
        raise ValueError(f"Architecture không được hỗ trợ: {architecture}")
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    model.to(device).eval()
    metadata = {key: value for key, value in checkpoint.items() if key != "model_state_dict"}
    return model, metadata


def print_checkpoint_status(path: Path) -> None:
    print(f"{path} | exists={path.exists()} | size={path.stat().st_size / (1024 ** 2):.2f} MB")


## 10. Train baseline tuần tự, đánh giá rồi giải phóng VRAM


In [ ]:
BASELINE_BEST_PATH = OUTPUT_DIR / "densenet121_best.pth"
BASELINE_FINAL_PATH = OUTPUT_DIR / "densenet121_final.pth"
BASELINE_HISTORY_PATH = OUTPUT_DIR / "densenet121_history.csv"

set_seed(SEED)
baseline_train_loader, baseline_val_loader, baseline_test_loader = create_dataloaders(
    EFFECTIVE_BATCH_SIZE, SEED
)
baseline_model = DenseNet121Baseline(pretrained=True).to(DEVICE)
baseline_total_params, baseline_initial_trainable_params = count_parameters(baseline_model)
baseline_training = train_model(
    model=baseline_model,
    model_name="DenseNet121",
    architecture="DenseNet121Baseline",
    train_loader=baseline_train_loader,
    val_loader=baseline_val_loader,
    best_checkpoint_path=BASELINE_BEST_PATH,
    history_path=BASELINE_HISTORY_PATH,
    coordinate_attention=False,
)
baseline_evaluation = save_and_evaluate_best(
    baseline_model,
    BASELINE_BEST_PATH,
    BASELINE_FINAL_PATH,
    baseline_test_loader,
    baseline_training,
)
print("Baseline test metrics:", {k: baseline_evaluation[k] for k in ("loss", "accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_f1")})
print(classification_report(
    baseline_evaluation["true_labels"], baseline_evaluation["predicted_labels"],
    labels=list(range(NUM_CLASSES)), target_names=class_names, zero_division=0
))
print("Baseline confusion matrix:\n", baseline_evaluation["confusion_matrix"])
print_checkpoint_status(BASELINE_BEST_PATH)
print_checkpoint_status(BASELINE_FINAL_PATH)

baseline_model = baseline_model.cpu()
del baseline_model, baseline_train_loader, baseline_val_loader, baseline_test_loader
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 11. Train DenseNet121-CA tuần tự và đánh giá trên cùng test split


In [ ]:
CA_BEST_PATH = OUTPUT_DIR / "densenet121_ca_best.pth"
CA_FINAL_PATH = OUTPUT_DIR / "densenet121_ca_final.pth"
CA_HISTORY_PATH = OUTPUT_DIR / "densenet121_ca_history.csv"

set_seed(SEED)
ca_train_loader, ca_val_loader, ca_test_loader = create_dataloaders(EFFECTIVE_BATCH_SIZE, SEED)
ca_model = DenseNet121WithCA(pretrained=True, reduction=CA_REDUCTION).to(DEVICE)
ca_total_params, ca_initial_trainable_params = count_parameters(ca_model)
ca_training = train_model(
    model=ca_model,
    model_name="DenseNet121 + CA",
    architecture="DenseNet121WithCA",
    train_loader=ca_train_loader,
    val_loader=ca_val_loader,
    best_checkpoint_path=CA_BEST_PATH,
    history_path=CA_HISTORY_PATH,
    coordinate_attention=True,
)
ca_evaluation = save_and_evaluate_best(
    ca_model, CA_BEST_PATH, CA_FINAL_PATH, ca_test_loader, ca_training
)
print("CA test metrics:", {k: ca_evaluation[k] for k in ("loss", "accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_f1")})
print(classification_report(
    ca_evaluation["true_labels"], ca_evaluation["predicted_labels"],
    labels=list(range(NUM_CLASSES)), target_names=class_names, zero_division=0
))
print("CA confusion matrix:\n", ca_evaluation["confusion_matrix"])
print_checkpoint_status(CA_BEST_PATH)
print_checkpoint_status(CA_FINAL_PATH)

ca_model = ca_model.cpu()
del ca_model, ca_train_loader, ca_val_loader
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 12. Bảng so sánh, deltas và kết luận tự động


In [ ]:
comparison_rows = [
    {
        "Model": "DenseNet121",
        "Best Epoch": baseline_training["best_epoch"],
        "Best Val Loss": baseline_training["best_val_loss"],
        "Best Val Accuracy": baseline_training["best_val_accuracy"],
        "Best Val Macro F1": baseline_training["best_val_macro_f1"],
        "Test Loss": baseline_evaluation["loss"],
        "Test Accuracy": baseline_evaluation["accuracy"],
        "Test Macro Precision": baseline_evaluation["macro_precision"],
        "Test Macro Recall": baseline_evaluation["macro_recall"],
        "Test Macro F1": baseline_evaluation["macro_f1"],
        "Test Weighted F1": baseline_evaluation["weighted_f1"],
        "Total Parameters": baseline_total_params,
        "Trainable Parameters": baseline_total_params,
        "Additional Parameters vs Baseline": 0,
        "Training Time (minutes)": baseline_training["training_time_minutes"],
        "Best Checkpoint": str(BASELINE_BEST_PATH),
        "Final Checkpoint": str(BASELINE_FINAL_PATH),
    },
    {
        "Model": "DenseNet121 + CA",
        "Best Epoch": ca_training["best_epoch"],
        "Best Val Loss": ca_training["best_val_loss"],
        "Best Val Accuracy": ca_training["best_val_accuracy"],
        "Best Val Macro F1": ca_training["best_val_macro_f1"],
        "Test Loss": ca_evaluation["loss"],
        "Test Accuracy": ca_evaluation["accuracy"],
        "Test Macro Precision": ca_evaluation["macro_precision"],
        "Test Macro Recall": ca_evaluation["macro_recall"],
        "Test Macro F1": ca_evaluation["macro_f1"],
        "Test Weighted F1": ca_evaluation["weighted_f1"],
        "Total Parameters": ca_total_params,
        "Trainable Parameters": ca_total_params,
        "Additional Parameters vs Baseline": ca_total_params - baseline_total_params,
        "Training Time (minutes)": ca_training["training_time_minutes"],
        "Best Checkpoint": str(CA_BEST_PATH),
        "Final Checkpoint": str(CA_FINAL_PATH),
    },
]
comparison_df = pd.DataFrame(comparison_rows)
comparison_csv_path = OUTPUT_DIR / "model_comparison.csv"
comparison_json_path = OUTPUT_DIR / "model_comparison.json"
comparison_df.to_csv(comparison_csv_path, index=False)

accuracy_delta = ca_evaluation["accuracy"] - baseline_evaluation["accuracy"]
macro_f1_delta = ca_evaluation["macro_f1"] - baseline_evaluation["macro_f1"]
parameter_delta = ca_total_params - baseline_total_params
deltas = {
    "accuracy_delta": accuracy_delta,
    "macro_f1_delta": macro_f1_delta,
    "parameter_delta": parameter_delta,
}
with comparison_json_path.open("w", encoding="utf-8") as file:
    json.dump({"models": comparison_rows, "deltas": deltas}, file, indent=2, ensure_ascii=False)

display(comparison_df)
print("Deltas:", deltas)
if macro_f1_delta > 0:
    print(f"Kết luận: CA cải thiện test macro-F1 {macro_f1_delta:+.6f} và accuracy {accuracy_delta:+.6f}.")
elif macro_f1_delta < 0:
    print(f"Kết luận: CA không cải thiện trong thí nghiệm này; test macro-F1 thay đổi {macro_f1_delta:+.6f}, accuracy {accuracy_delta:+.6f}.")
else:
    print(f"Kết luận: CA không thay đổi test macro-F1; accuracy thay đổi {accuracy_delta:+.6f}.")


## 13. Biểu đồ và confusion matrices


In [ ]:
baseline_history_df = pd.DataFrame(baseline_training["history"])
ca_history_df = pd.DataFrame(ca_training["history"])


def plot_two_curves(column: str, ylabel: str, title: str, output_path: Path) -> None:
    plt.figure(figsize=(9, 5))
    plt.plot(baseline_history_df["epoch"], baseline_history_df[column], label="DenseNet121")
    plt.plot(ca_history_df["epoch"], ca_history_df[column], label="DenseNet121 + CA")
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.show()


fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for frame, label in ((baseline_history_df, "DenseNet121"), (ca_history_df, "DenseNet121 + CA")):
    axes[0].plot(frame["epoch"], frame["train_loss"], label=label)
    axes[1].plot(frame["epoch"], frame["val_loss"], label=label)
axes[0].set_title("Train loss"); axes[1].set_title("Validation loss")
for axis in axes:
    axis.set_xlabel("Epoch"); axis.set_ylabel("Loss"); axis.grid(alpha=0.3); axis.legend()
fig.tight_layout(); fig.savefig(FIGURES_DIR / "loss_comparison.png", dpi=200, bbox_inches="tight"); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for frame, label in ((baseline_history_df, "DenseNet121"), (ca_history_df, "DenseNet121 + CA")):
    axes[0].plot(frame["epoch"], frame["train_accuracy"], label=label)
    axes[1].plot(frame["epoch"], frame["val_accuracy"], label=label)
axes[0].set_title("Train accuracy"); axes[1].set_title("Validation accuracy")
for axis in axes:
    axis.set_xlabel("Epoch"); axis.set_ylabel("Accuracy"); axis.grid(alpha=0.3); axis.legend()
fig.tight_layout(); fig.savefig(FIGURES_DIR / "accuracy_comparison.png", dpi=200, bbox_inches="tight"); plt.show()

plot_two_curves(
    "val_macro_f1", "Macro-F1", "Validation macro-F1",
    FIGURES_DIR / "val_macro_f1_comparison.png"
)

metric_plot = pd.DataFrame({
    "Model": ["DenseNet121", "DenseNet121 + CA"],
    "Test Accuracy": [baseline_evaluation["accuracy"], ca_evaluation["accuracy"]],
    "Test Macro-F1": [baseline_evaluation["macro_f1"], ca_evaluation["macro_f1"]],
}).set_index("Model")
metric_plot.plot(kind="bar", figsize=(9, 5), rot=0)
plt.ylim(0, 1); plt.ylabel("Score"); plt.title("Test metrics comparison")
plt.grid(axis="y", alpha=0.3); plt.tight_layout()
plt.savefig(FIGURES_DIR / "test_metrics_comparison.png", dpi=200, bbox_inches="tight"); plt.show()


def save_confusion_matrix(matrix: np.ndarray, title: str, path: Path) -> None:
    plt.figure(figsize=(8, 6))
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(title); plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight"); plt.show()


save_confusion_matrix(
    baseline_evaluation["confusion_matrix"], "DenseNet121 confusion matrix",
    FIGURES_DIR / "confusion_matrix_densenet121.png"
)
save_confusion_matrix(
    ca_evaluation["confusion_matrix"], "DenseNet121 + CA confusion matrix",
    FIGURES_DIR / "confusion_matrix_densenet121_ca.png"
)


## 14. Bắt buộc reload hai final checkpoint và inference cùng một batch


In [ ]:
baseline_loaded, baseline_metadata = load_trained_model(BASELINE_FINAL_PATH, DEVICE)
ca_loaded, ca_metadata = load_trained_model(CA_FINAL_PATH, DEVICE)

reload_images, _ = next(iter(ca_test_loader))
reload_images = reload_images.to(DEVICE)
with torch.no_grad():
    baseline_reload_output = baseline_loaded(reload_images)
    ca_reload_output = ca_loaded(reload_images)
actual_batch_size = reload_images.size(0)
assert baseline_reload_output.shape == (actual_batch_size, NUM_CLASSES)
assert ca_reload_output.shape == (actual_batch_size, NUM_CLASSES)
print("Reload baseline output shape:", tuple(baseline_reload_output.shape))
print("Reload CA output shape:", tuple(ca_reload_output.shape))
print("Cả hai final checkpoint đã reload và inference thành công.")

baseline_loaded = baseline_loaded.cpu()
ca_loaded = ca_loaded.cpu()
del baseline_loaded, ca_loaded, reload_images, baseline_reload_output, ca_reload_output, ca_test_loader
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 15. Kiểm tra artifacts và tạo ZIP để tải từ Kaggle


In [ ]:
required_files = [
    BASELINE_BEST_PATH, BASELINE_FINAL_PATH, CA_BEST_PATH, CA_FINAL_PATH,
    BASELINE_HISTORY_PATH, CA_HISTORY_PATH,
    comparison_csv_path, comparison_json_path,
]
for path in required_files:
    if not path.exists():
        raise FileNotFoundError(f"Thiếu artifact bắt buộc: {path}")
    print_checkpoint_status(path)

temporary_zip_base = Path("/kaggle/working/DenseNet121_vs_CA_results")
temporary_zip_path = Path(shutil.make_archive(str(temporary_zip_base), "zip", root_dir=OUTPUT_DIR))
zip_path = OUTPUT_DIR / "DenseNet121_vs_CA_results.zip"
shutil.move(str(temporary_zip_path), str(zip_path))
print_checkpoint_status(zip_path)

print("\nToàn bộ file trong OUTPUT_DIR:")
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(path, "-", path.stat().st_size / (1024 ** 2), "MB")
